In [ ]:
cell_start(0, 'Setup e Autenticacao')import os, sys, json, subprocess, time, io, gc, shutil, globfrom google.oauth2.credentials import Credentialsfrom google.auth.transport.requests import Requestfrom googleapiclient.discovery import buildfrom googleapiclient.http import MediaFileUpload, MediaIoBaseDownloadimport requests as http_requestsNOTEBOOK_NAME = "merge-final"STEP_NAME = "step_merge"print("Instalando dependencias...")os.system("apt-get install -y ffmpeg > /dev/null 2>&1")def _load_secrets():    try:        from kaggle_secrets import UserSecretsClient        _s = UserSecretsClient()        def _get(name):            try: return _s.get_secret(name)            except: return ""        print("Carregando Kaggle Secrets...")        return _get    except ImportError:        from dotenv import load_dotenv; load_dotenv()        return lambda name: os.getenv(name, "")_get = _load_secrets()DRIVE_REFRESH_TOKEN = _get("DRIVE_REFRESH_TOKEN")DRIVE_CLIENT_ID = _get("DRIVE_CLIENT_ID")DRIVE_CLIENT_SECRET = _get("DRIVE_CLIENT_SECRET")HF_TOKEN = _get("HF_TOKEN")DATABASE_URL = _get("DATABASE_URL")PROJECT_ID = _get("PIPELINE_PROJECT_ID")PIPELINE_WEBHOOK_URL = _get("PIPELINE_WEBHOOK_URL")print("Autenticando Drive...")try:    _creds = Credentials(token=None, refresh_token=DRIVE_REFRESH_TOKEN,        token_uri="https://oauth2.googleapis.com/token",        client_id=DRIVE_CLIENT_ID, client_secret=DRIVE_CLIENT_SECRET,        scopes=["https://www.googleapis.com/auth/drive"])    _creds.refresh(Request())    drive_service = build("drive", "v3", credentials=_creds)    print("Drive autenticado!")except Exception as e:    drive_service = None    print(f"Falha Drive: {e}")def _buscar_id(caminho):    partes = caminho.strip("/").split("/")    pid = "root"    for p in partes:        q = f"name='{p}' and '{pid}' in parents and trashed=false"        r = drive_service.files().list(q=q, fields="files(id,mimeType)").execute()        a = r.get("files", [])        if not a: return None        pid = a[0]["id"]    return piddef _garantir_pasta(caminho):    partes = caminho.strip("/").split("/")    pid = "root"    for p in partes:        q = f"name='{p}' and '{pid}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"        r = drive_service.files().list(q=q, fields="files(id)").execute()        e = r.get("files", [])        if e: pid = e[0]["id"]        else:            nova = drive_service.files().create(body={"name": p, "mimeType": "application/vnd.google-apps.folder", "parents": [pid]}, fields="id").execute()            pid = nova["id"]    return piddef baixar_do_drive(caminho_drive, destino_local):    if os.path.exists(destino_local): return True    try:        fid = _buscar_id(caminho_drive)        if not fid: print(f"  Nao encontrado: {caminho_drive}"); return False        req = drive_service.files().get_media(fileId=fid)        os.makedirs(os.path.dirname(destino_local) or ".", exist_ok=True)        with open(destino_local, "wb") as fh:            dl = MediaIoBaseDownload(fh, req); done = False            while not done: _, done = dl.next_chunk()        print(f"  Baixado: {caminho_drive}"); return True    except Exception as ex: print(f"  Erro: {caminho_drive}: {ex}"); return Falsedef salvar_no_drive(caminho_local, caminho_drive):    if not drive_service or not os.path.exists(caminho_local): return    try:        partes = caminho_drive.strip("/").split("/")        nome = partes[-1]; pasta = "/".join(partes[:-1]) if len(partes) > 1 else ""        pid = _garantir_pasta(pasta) if pasta else "root"        q = f"name='{nome}' and '{pid}' in parents and trashed=false"        r = drive_service.files().list(q=q, fields="files(id)").execute()        e = r.get("files", []); media = MediaFileUpload(caminho_local, resumable=True)        if e: drive_service.files().update(fileId=e[0]["id"], media_body=media).execute()        else: drive_service.files().create(body={"name": nome, "parents": [pid]}, media_body=media, fields="id").execute()        print(f"  Salvo: {caminho_drive}")    except Exception as ex: print(f"  Erro salvar {caminho_drive}: {ex}")_cell_timers = {}def _db_exec(query, params):    if not DATABASE_URL: return    try:        import psycopg2; conn = psycopg2.connect(DATABASE_URL); cur = conn.cursor()        cur.execute(query, params); conn.commit(); cur.close(); conn.close()    except: passdef cell_start(idx, name=""):    _cell_timers[idx] = time.time()    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")    if not PROJECT_ID: return    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,'running',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))    _db_exec("UPDATE pipeline_cell_tracking SET status='running',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))def cell_end(idx, status="done", msg=""):    elapsed = ""    if idx in _cell_timers:        s = int(time.time() - _cell_timers.pop(idx))        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"    icon = "OK" if status == "done" else "ERRO"    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'─'*50}")    if not PROJECT_ID: return    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))def report_step(status, msg=""):    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")    if not PROJECT_ID: return    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))    _db_exec("INSERT INTO pipeline_logs (project_id,step,status,message) VALUES (%s::uuid,%s,%s,%s)", (PROJECT_ID, STEP_NAME, status, msg))DRIVE_ATIVO = "KAGGLE/PIPELINE/ATIVO"DRIVE_WATERMARK = "KAGGLE/PIPELINE/WATERMARK"DRIVE_ENHANCER = "KAGGLE/PIPELINE/ENHANCER"DRIVE_OMNI = "KAGGLE/PIPELINE/OMNI"DRIVE_RENDER = "KAGGLE/PIPELINE/RENDER"DRIVE_FINAL = "KAGGLE/PIPELINE/FINAL"BASE_PATH = "/kaggle/working"os.makedirs(BASE_PATH, exist_ok=True)cell_end(0, "done", "Setup concluido")

In [ ]:
cell_start(1, 'Download das Partes')baixar_do_drive(f"{DRIVE_RENDER}/pt1_renderizado.mp4", f"{BASE_PATH}/pt1_renderizado.mp4")baixar_do_drive(f"{DRIVE_RENDER}/pt2_renderizado.mp4", f"{BASE_PATH}/pt2_renderizado.mp4")print("Partes baixadas!")cell_end(1, 'done', 'Download das Partes concluido')

In [ ]:
cell_start(2, 'Merge Final')OUTPUT = f"{BASE_PATH}/video_final.mp4"concat_file = f"{BASE_PATH}/merge_concat.txt"with open(concat_file, "w") as f:    f.write(f"file '{BASE_PATH}/pt1_renderizado.mp4'\n")    f.write(f"file '{BASE_PATH}/pt2_renderizado.mp4'\n")subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",concat_file,"-c","copy",OUTPUT], check=True, capture_output=True)print(f"  Merge concluido: {OUTPUT}")cell_end(2, 'done', 'Merge Final concluido')

In [ ]:
cell_start(3, 'Upload Final')salvar_no_drive(f"{BASE_PATH}/video_final.mp4", f"{DRIVE_FINAL}/video_final.mp4")report_step("done", "Merge final concluido")print("VIDEO FINAL PRONTO!")cell_end(3, 'done', 'Upload Final concluido')